In [ ]:
# ============================================================================
# ETAPA 5 — MULTI-HEAD COM SPLIT ESPACIAL
# ============================================================================
# Objetivo: Re-treinar o Multi-Head usando o mesmo split espacial
# por hexágono do etapa4. Comparação válida com o baseline:
#   - Mesmos dados (treino_balanceado_FINAL.csv)
#   - Mesmo split (spatial_split_{train/val/test}.csv)
#   - Mesmos hiperparâmetros (lr, batch, early stopping)
#   - Única variável: arquitetura (Multi-Head vs single-head)
#
# Referências:
#   - Baseline (etapa4): val=65.0%, test=61.3%, F1=0.593, gap=-3.7pp
#   - Multi-Head original (etapa3, random split): test=68.2%, F1=0.619
#     [NÃO comparável diretamente — split diferente]
#
# Freeze: multihead_spatial_frozen/
# ============================================================================

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import json
from datetime import datetime
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# Referência do baseline para comparação imediata
BASELINE_SPATIAL = {
    'val_accuracy':  0.6501,
    'test_accuracy': 0.6126,
    'test_f1':       0.5926,
    'test_auc':      0.6547,
    'val_test_gap':  -0.0374
}

print(f"✅ Configuração OK!")
print(f"   Device:   {DEVICE}")
print(f"   PyTorch:  {torch.__version__}")
print(f"   Notebook: ETAPA 5 — Multi-Head + Split Espacial")
print()
print(f"📊 Referência baseline (etapa4, mesmo split):")
print(f"   Test Accuracy: {BASELINE_SPATIAL['test_accuracy']:.1%}")
print(f"   Test F1:       {BASELINE_SPATIAL['test_f1']:.3f}")
print(f"   Val-Test Gap:  {BASELINE_SPATIAL['val_test_gap']*100:.1f}pp")

In [ ]:
# ============================================================================
# MULTI-HEAD ARCHITECTURE — idêntica ao etapa3
# ============================================================================

class MultiHeadSpatialModel(nn.Module):
    """Multi-Head com soft gating. Idêntico ao etapa3."""

    def __init__(self, input_dim=287, hidden_dim=256, encoder_dim=128,
                 head_hidden_dim=64, dropout=0.3):
        super().__init__()

        # Shared Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, encoder_dim), nn.ReLU(), nn.Dropout(dropout)
        )

        # Head Disperso
        self.head_disperso = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1), nn.Sigmoid()
        )

        # Head Cluster
        self.head_cluster = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden_dim), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1), nn.Sigmoid()
        )

        # Gate Network
        gate_input_dim = encoder_dim + 3   # 128 + 3 spatial features
        self.gate = nn.Sequential(
            nn.Linear(gate_input_dim, 32), nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(32, 2), nn.Softmax(dim=1)
        )

    def forward(self, x, spatial_features):
        encoded       = self.encoder(x)
        pred_disperso = self.head_disperso(encoded)
        pred_cluster  = self.head_cluster(encoded)

        gate_input   = torch.cat([encoded, spatial_features], dim=1)
        gate_weights = self.gate(gate_input)

        w_disperso = gate_weights[:, 0:1]
        w_cluster  = gate_weights[:, 1:2]

        prediction = w_disperso * pred_disperso + w_cluster * pred_cluster
        return prediction, gate_weights


# Verificação
model = MultiHeadSpatialModel().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

dummy_x  = torch.randn(4, 287).to(DEVICE)
dummy_sp = torch.zeros(4, 3).to(DEVICE)
out, gw  = model(dummy_x, dummy_sp)

print(f"✅ MultiHeadSpatialModel criado!")
print(f"   Parâmetros: {total_params:,}")
print(f"   Output shape: {out.shape}  (esperado: [4, 1])")
print(f"   Gate shape:   {gw.shape}   (esperado: [4, 2])")
print(f"   Gate sum:     {gw.sum(dim=1).mean():.4f} (esperado: 1.0000)")
print()

# Comparação de capacidade com o baseline
baseline_params = 287*256 + 256 + 256*128 + 128 + 128*64 + 64 + 64*1 + 1
print(f"   Comparação de parâmetros:")
print(f"     Baseline MLP:         ~{baseline_params:,}")
print(f"     Multi-Head:            {total_params:,}")
print(f"     Overhead:             +{(total_params/baseline_params - 1)*100:.0f}%")

In [ ]:
# ============================================================================
# DATASET — idêntico ao etapa3 e etapa4
# ============================================================================

import rasterio
from rasterio.windows import Window

DATA_DIR      = r"D:\Projetos\Cerrado\LULC"
PATTERN       = "brazil_coverage_{ano}_Cerrado.tif"
YEARS         = list(range(1985, 2025))
NODATA_IN     = 255
NODATA_OUT    = 0
PATCH_N       = 7
MAX_SERIE_LEN = len(YEARS) - 1
PATCH_YEARS   = 5


def _path(year, data_dir=DATA_DIR, pattern=PATTERN):
    return os.path.join(data_dir, pattern.format(ano=year))

def _ler_pixel(year, row, col, data_dir=DATA_DIR, pattern=PATTERN):
    with rasterio.open(_path(year, data_dir, pattern)) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def _ler_patch(year, row, col, n, data_dir=DATA_DIR, pattern=PATTERN):
    half = n // 2
    with rasterio.open(_path(year, data_dir, pattern)) as ds:
        H, W = ds.height, ds.width
        col0 = min(max(0, col - half), W - n)
        row0 = min(max(0, row - half), H - n)
        arr  = ds.read(1, window=Window(col0, row0, n, n), out_dtype="uint8")
    return np.where(arr == NODATA_IN, NODATA_OUT, arr).astype(np.uint8)


class LULCMultiHeadDataset(Dataset):

    def __init__(self, csv_path, indices=None,
                 data_dir=DATA_DIR, pattern=PATTERN,
                 years=YEARS, patch_n=PATCH_N,
                 max_serie_len=MAX_SERIE_LEN,
                 patch_years=PATCH_YEARS):

        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['label'].notna()].reset_index(drop=True)
        if indices is not None:
            self.df = self.df.iloc[indices].reset_index(drop=True)

        self.data_dir      = data_dir
        self.pattern       = pattern
        self.years         = years
        self.patch_n       = patch_n
        self.max_serie_len = max_serie_len
        self.patch_years   = patch_years

        print(f"  Dataset: {len(self.df):,} samples")
        print(f"  Labels: {self.df['label'].value_counts().to_dict()}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r     = self.df.iloc[idx]
        row   = int(r["row"])
        col   = int(r["col"])
        T     = int(r["T"])
        label = int(r["label"])

        anos_serie = [y for y in self.years if y < T]
        if len(anos_serie) > 0:
            serie_raw = np.array(
                [_ler_pixel(y, row, col, self.data_dir, self.pattern)
                 for y in anos_serie], dtype=np.float32)
            serie_raw = np.where(serie_raw == NODATA_IN,
                                 NODATA_OUT, serie_raw).astype(np.float32)
        else:
            serie_raw = np.array([], dtype=np.float32)

        serie_len = len(serie_raw)
        serie     = np.zeros(self.max_serie_len, dtype=np.float32)
        if serie_len > 0:
            serie[self.max_serie_len - serie_len:] = serie_raw

        anos_patch = [y for y in self.years if y < T][-self.patch_years:]
        patch = np.zeros((self.patch_years, self.patch_n, self.patch_n),
                         dtype=np.float32)
        for i, y in enumerate(anos_patch):
            patch_idx = self.patch_years - len(anos_patch) + i
            patch[patch_idx] = _ler_patch(y, row, col, self.patch_n,
                                          self.data_dir, self.pattern)

        if serie_len > 0:
            has_21    = float(np.sum(serie_raw == 21)) / serie_len
            anos_past = 0
            for v in reversed(serie_raw):
                if v == 15: anos_past += 1
                else:       break
            cl_tm1 = float(serie_raw[-1])
        else:
            has_21 = anos_past = cl_tm1 = 0.0

        aux_features = np.array([has_21, float(anos_past), cl_tm1],
                                dtype=np.float32)

        patch_flat = patch.flatten()
        features   = np.concatenate([serie, patch_flat, aux_features])
        assert features.shape == (287,), \
            f"Features shape errado: {features.shape} (idx={idx}, T={T})"

        if 'pattern_code' in self.df.columns and pd.notna(r.get('pattern_code')):
            pattern = str(r['pattern_code'])
            is_cluster_conv = 1.0 if pattern.startswith('CC') else 0.0
            is_cluster_use  = 1.0 if 'C' in pattern[-2:] else 0.0
            consol_level    = 2.0 if pattern.startswith('CC') else 1.0
        else:
            is_cluster_conv = is_cluster_use = 0.5
            consol_level    = 1.5

        spatial = np.array([is_cluster_conv, is_cluster_use, consol_level],
                           dtype=np.float32)

        return (
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(spatial,  dtype=torch.float32),
            torch.tensor([label],  dtype=torch.float32)
        )


print("✅ LULCMultiHeadDataset definido")

In [ ]:
# ============================================================================
# CARREGAR DADOS — mesmo split do etapa4
# ============================================================================

BASE_DIR  = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
SPLIT_DIR = BASE_DIR / "spatial_split"

print("Carregando splits espaciais (idênticos ao etapa4)...")
print()

print("Train:")
train_dataset = LULCMultiHeadDataset(SPLIT_DIR / "spatial_split_train.csv")
print()
print("Val:")
val_dataset   = LULCMultiHeadDataset(SPLIT_DIR / "spatial_split_val.csv")
print()
print("Test:")
test_dataset  = LULCMultiHeadDataset(SPLIT_DIR / "spatial_split_test.csv")

print(f"\n✅ Splits carregados (idênticos ao etapa4):")
print(f"   Train: {len(train_dataset):,}")
print(f"   Val:   {len(val_dataset):,}")
print(f"   Test:  {len(test_dataset):,}")

BATCH_SIZE = 256

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

print(f"\n✅ DataLoaders prontos! Batch size: {BATCH_SIZE}")

# Sanity check
print("\n🔍 Testando extração de features...")
sample_features, sample_spatial, sample_label = train_dataset[0]
assert sample_features.shape == (287,), f"Erro: {sample_features.shape}"
print(f"   Features: {sample_features.shape}  ✅")
print(f"   Spatial:  {sample_spatial.shape}   ✅")
print(f"   Label:    {sample_label.item()}")

In [ ]:
# ============================================================================
# FUNÇÕES DE TREINO — idênticas ao etapa3 e etapa4
# ============================================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = correct = total = 0

    pbar = tqdm(loader, desc="Training")
    for features, spatial, labels in pbar:
        features = features.to(device)
        spatial  = spatial.to(device)
        labels   = labels.to(device)

        predictions, _ = model(features, spatial)
        loss = criterion(predictions, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred_class  = (predictions > 0.5).float()
        correct    += (pred_class == labels).sum().item()
        total      += labels.size(0)

        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'acc':  f'{100*correct/total:.1f}%'})

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device, collect_gate=False):
    """Avalia modelo. Se collect_gate=True, retorna gate weights também."""
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels, all_gates = [], [], []

    with torch.no_grad():
        for features, spatial, labels in tqdm(loader, desc="Evaluating"):
            features = features.to(device)
            spatial  = spatial.to(device)
            labels   = labels.to(device)

            predictions, gate_weights = model(features, spatial)
            loss = criterion(predictions, labels)

            total_loss += loss.item()
            pred_class  = (predictions > 0.5).float()
            correct    += (pred_class == labels).sum().item()
            total      += labels.size(0)

            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            if collect_gate and gate_weights is not None:
                all_gates.extend(gate_weights.cpu().numpy())

    preds_np      = np.array(all_preds).flatten()
    labels_np     = np.array(all_labels).flatten()
    pred_class_np = (preds_np > 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_np, pred_class_np, average='binary')
    try:
        auc = roc_auc_score(labels_np, preds_np)
    except:
        auc = 0.0

    metrics = {
        'loss':      total_loss / len(loader),
        'accuracy':  correct / total,
        'precision': precision,
        'recall':    recall,
        'f1':        f1,
        'auc':       auc
    }

    gate_np = np.array(all_gates) if all_gates else None
    return metrics, gate_np


print("✅ Funções de treino definidas!")

In [ ]:
# ============================================================================
# TREINO — hiperparâmetros idênticos ao etapa4
# ============================================================================

torch.manual_seed(SEED)
model = MultiHeadSpatialModel().to(DEVICE)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True)

N_EPOCHS       = 30
EARLY_STOP_PAT = 10

best_val_loss    = float('inf')
best_model_state = None
best_val_metrics = None
patience_counter = 0
history          = {'train_loss': [], 'train_acc': [], 'val_metrics': []}

print("=" * 70)
print("INICIANDO TREINO — MULTI-HEAD (SPLIT ESPACIAL)")
print("=" * 70)
print(f"  Modelo:    MultiHeadSpatialModel "
      f"({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Split:     Espacial por hexágono (idêntico ao etapa4)")
print(f"  Train:     {len(train_dataset):,} | Val: {len(val_dataset):,} "
      f"| Test: {len(test_dataset):,}")
print(f"  Epochs:    {N_EPOCHS} (early stop patience={EARLY_STOP_PAT})")
print(f"  LR:        0.001 (ReduceLROnPlateau patience=5)")
print(f"  Device:    {DEVICE}")
print()
print(f"  [Baseline etapa4] test={BASELINE_SPATIAL['test_accuracy']:.1%}, "
      f"F1={BASELINE_SPATIAL['test_f1']:.3f}, "
      f"gap={BASELINE_SPATIAL['val_test_gap']*100:.1f}pp")
print()

for epoch in range(N_EPOCHS):
    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    print("-" * 70)

    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, DEVICE)
    val_metrics, _ = evaluate(
        model, val_loader, criterion, DEVICE, collect_gate=False)

    scheduler.step(val_metrics['loss'])

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_metrics'].append(val_metrics)

    print(f"\nTrain - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val   - Loss: {val_metrics['loss']:.4f}, "
          f"Acc: {val_metrics['accuracy']:.4f}, "
          f"F1: {val_metrics['f1']:.4f}, "
          f"AUC: {val_metrics['auc']:.4f}")

    if val_metrics['loss'] < best_val_loss:
        best_val_loss    = val_metrics['loss']
        best_model_state = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
        best_val_metrics = val_metrics.copy()
        patience_counter = 0
        print("  → Best model updated!")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PAT:
            print(f"\nEarly stopping após {epoch+1} epochs.")
            break

    print()

print("=" * 70)
print("TREINO CONCLUÍDO")
print("=" * 70)
print(f"  Epochs rodadas: {len(history['train_loss'])}")
print(f"  Melhor val loss: {best_val_loss:.4f}")
print(f"  Melhor val acc:  {best_val_metrics['accuracy']:.4f}")
print(f"  Melhor val F1:   {best_val_metrics['f1']:.4f}")

In [ ]:
# ============================================================================
# AVALIAÇÃO FINAL — TEST SET + GATE WEIGHTS
# ============================================================================

model.load_state_dict(best_model_state)
model.to(DEVICE)

print("=" * 70)
print("AVALIAÇÃO FINAL — TEST SET (SPLIT ESPACIAL)")
print("=" * 70)

test_metrics, gate_weights_np = evaluate(
    model, test_loader, criterion, DEVICE, collect_gate=True)

val_acc      = best_val_metrics['accuracy']
test_acc     = test_metrics['accuracy']
val_test_gap = test_acc - val_acc

# ── Tabela de métricas ────────────────────────────────────────────────────────
print(f"\n{'Métrica':<15} {'Validation':>12} {'Test':>12}")
print("-" * 42)
for k in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    print(f"{k:<15} {best_val_metrics[k]:>12.4f} {test_metrics[k]:>12.4f}")
print(f"\nVal-Test Gap: {val_test_gap*100:+.2f}pp")

# ── Gate weights ──────────────────────────────────────────────────────────────
if gate_weights_np is not None:
    print(f"\n{'='*70}")
    print("GATE WEIGHTS — TEST SET")
    print(f"{'='*70}")
    print(f"  Head Disperso: mean={gate_weights_np[:,0].mean():.3f}, "
          f"std={gate_weights_np[:,0].std():.3f}, "
          f"range=[{gate_weights_np[:,0].min():.3f}, "
          f"{gate_weights_np[:,0].max():.3f}]")
    print(f"  Head Cluster:  mean={gate_weights_np[:,1].mean():.3f}, "
          f"std={gate_weights_np[:,1].std():.3f}, "
          f"range=[{gate_weights_np[:,1].min():.3f}, "
          f"{gate_weights_np[:,1].max():.3f}]")

    pct_balanced = np.mean(
        (gate_weights_np[:,0] > 0.4) & (gate_weights_np[:,0] < 0.6)
    ) * 100
    print(f"  Amostras com pesos balanceados (0.4<w<0.6): {pct_balanced:.1f}%")
    print(f"  Comportamento: {'Soft gating (ensemble)' if gate_weights_np[:,0].std() < 0.15 else 'Hard gating (especialização)'}")

# ── Comparação com baseline ───────────────────────────────────────────────────
print(f"\n{'='*70}")
print("COMPARAÇÃO MULTI-HEAD vs BASELINE (split espacial)")
print(f"{'='*70}")
print(f"{'Métrica':<20} {'Baseline':>10} {'Multi-Head':>12} {'Delta':>8}")
print("-" * 54)

comparisons = [
    ('Test Accuracy',  BASELINE_SPATIAL['test_accuracy'],  test_acc),
    ('Test F1',        BASELINE_SPATIAL['test_f1'],        test_metrics['f1']),
    ('Test AUC',       BASELINE_SPATIAL['test_auc'],       test_metrics['auc']),
    ('Val-Test Gap',   BASELINE_SPATIAL['val_test_gap'],   val_test_gap),
]

for name, base, mh in comparisons:
    delta = mh - base
    arrow = '↑' if delta > 0.002 else '↓' if delta < -0.002 else '~'
    if name == 'Val-Test Gap':
        print(f"{name:<20} {base*100:>+9.1f}pp {mh*100:>+11.1f}pp "
              f"{delta*100:>+7.1f}pp")
    else:
        print(f"{name:<20} {base:>10.4f} {mh:>12.4f} "
              f"{delta:>+7.4f} {arrow}")

In [ ]:
# ============================================================================
# CONGELAR MODELO MULTI-HEAD SPATIAL — VERSÃO FINAL
# ============================================================================

BASE_DIR   = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
FREEZE_DIR = BASE_DIR / "multihead_spatial_frozen"
FREEZE_DIR.mkdir(exist_ok=True, parents=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 70)
print("CONGELANDO MULTI-HEAD (SPLIT ESPACIAL)")
print("=" * 70)
print(f"  Timestamp:  {timestamp}")
print(f"  Diretório:  {FREEZE_DIR}")

# 1. Checkpoint .pth
checkpoint = {
    'model_state_dict': best_model_state,
    'model_class':      'MultiHeadSpatialModel',
    'config': {
        'input_dim':       287,
        'hidden_dim':      256,
        'encoder_dim':     128,
        'head_hidden_dim':  64,
        'dropout':         0.3
    },
    'metrics': {
        'validation': {k: float(v) for k, v in best_val_metrics.items()},
        'test':       {k: float(v) for k, v in test_metrics.items()},
        'val_test_gap': float(val_test_gap)
    },
    'training': {
        'n_epochs':   len(history['train_loss']),
        'batch_size': BATCH_SIZE,
        'lr':         0.001,
        'optimizer':  'Adam',
        'criterion':  'BCELoss',
        'seed':       SEED
    },
    'dataset': {
        'split_method':   'hexagon_stratified_by_Class8590',
        'split_metadata': str(BASE_DIR / 'spatial_split' /
                              'spatial_split_metadata.json'),
        'train_samples':  len(train_dataset),
        'val_samples':    len(val_dataset),
        'test_samples':   len(test_dataset),
        'spatial_overlap_pct': 0
    },
    'baseline_comparison': {
        'baseline_test_accuracy': BASELINE_SPATIAL['test_accuracy'],
        'multihead_test_accuracy': float(test_acc),
        'improvement_absolute':   float(test_acc - BASELINE_SPATIAL['test_accuracy']),
        'baseline_f1':            BASELINE_SPATIAL['test_f1'],
        'multihead_f1':           float(test_metrics['f1']),
        'f1_improvement':         float(test_metrics['f1'] - BASELINE_SPATIAL['test_f1']),
        'baseline_gap':           BASELINE_SPATIAL['val_test_gap'],
        'multihead_gap':          float(val_test_gap),
        'note': 'Valid comparison: same dataset, same spatial split, '
                'only variable is architecture'
    }
}

pth_path = FREEZE_DIR / f"multihead_spatial_frozen_{timestamp}.pth"
torch.save(checkpoint, pth_path)
size_mb = pth_path.stat().st_size / (1024**2)
print(f"\n✅ Modelo salvo: {pth_path.name} ({size_mb:.2f} MB)")

# 2. Gate weights
if gate_weights_np is not None:
    gate_path = FREEZE_DIR / f"gate_weights_{timestamp}.npy"
    np.save(gate_path, gate_weights_np)
    print(f"✅ Gate weights salvos: {gate_path.name} "
          f"(shape: {gate_weights_np.shape})")

# 3. JSON legível
results = {
    'model_info': {
        'name':        'Multi-Head Spatial Model',
        'version':     'v2.0-spatial-split',
        'frozen_date': timestamp,
        'architecture': 'Shared Encoder + Dual Heads + Soft Gate'
    },
    'performance': {
        'validation': {k: round(float(v), 4)
                       for k, v in best_val_metrics.items()},
        'test':       {k: round(float(v), 4)
                       for k, v in test_metrics.items()},
        'generalization': {
            'val_test_gap':   round(float(val_test_gap), 4),
            'gap_percentage': f"{val_test_gap*100:.2f}pp",
            'status':         'EXCELLENT' if abs(val_test_gap) < 0.05
                              else 'ACCEPTABLE' if abs(val_test_gap) < 0.10
                              else 'OVERFITTING'
        }
    },
    'baseline_comparison': checkpoint['baseline_comparison'],
    'gate_weights': {
        'behavior':      'Soft Gating' if gate_weights_np is not None
                         and gate_weights_np[:,0].std() < 0.15
                         else 'Hard Gating',
        'disperso_mean': round(float(gate_weights_np[:,0].mean()), 3)
                         if gate_weights_np is not None else None,
        'disperso_std':  round(float(gate_weights_np[:,0].std()), 3)
                         if gate_weights_np is not None else None,
        'cluster_mean':  round(float(gate_weights_np[:,1].mean()), 3)
                         if gate_weights_np is not None else None,
        'cluster_std':   round(float(gate_weights_np[:,1].std()), 3)
                         if gate_weights_np is not None else None,
    },
    'dataset_info': {
        'source_file':   'treino_balanceado_FINAL.csv',
        'split_method':  'hexagon_stratified_by_Class8590',
        'spatial_overlap_train_test': '0%',
        'splits': {
            'train': len(train_dataset),
            'val':   len(val_dataset),
            'test':  len(test_dataset)
        }
    }
}

json_path = FREEZE_DIR / f"results_{timestamp}.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"✅ Resultados salvos: {json_path.name}")

# 4. Resumo final
print("\n" + "=" * 70)
print("MULTI-HEAD SPATIAL — CONGELADO")
print("=" * 70)
print(f"\n📁 {FREEZE_DIR}")
print(f"\n📊 Performance (split espacial):")
print(f"  Val Accuracy:   {val_acc:.1%}")
print(f"  Test Accuracy:  {test_acc:.1%}")
print(f"  Test F1:        {test_metrics['f1']:.3f}")
print(f"  Test AUC:       {test_metrics['auc']:.3f}")
print(f"  Val-Test Gap:   {val_test_gap*100:+.1f}pp")
print(f"\n📊 vs Baseline (etapa4, mesmo split):")
delta_acc = test_acc - BASELINE_SPATIAL['test_accuracy']
delta_f1  = test_metrics['f1'] - BASELINE_SPATIAL['test_f1']
print(f"  Accuracy delta: {delta_acc*100:+.1f}pp")
print(f"  F1 delta:       {delta_f1:+.3f}")
print()
print(f"🎯 Próximo passo: Update 5 do OSF com estes resultados")